# Building a Strands Agent with Memory via Flotorch

This notebook demonstrates how to build an advanced Strands agent with both session-based and memory capabilities using the Flotorch platform. This allows the agent to recall information from the current conversation as well as from previous, separate interactions, enhanced by custom tools for arithmetic operations.

### Prerequisites
Configure your model, memory provider, and API key in the Flotorch console[](https://console.flotorch.cloud/).

### Viewing Logs
Logs can be viewed in the logs tab of the Flotorch console[](https://console.flotorch.cloud/).

### Key Concepts:
- **Memory**: Provided by `FlotorchStrandsMemory` for persistent knowledge across all sessions.
- **Custom Tools**: `multiply` and `addition` tools for performing arithmetic operations.

## 1. Setup and Imports

The following cells install the necessary packages, configure API credentials, and import required components from Flotorch and Strands, including those needed for memory and custom tools.

In [ ]:
%pip install flotorch[strands]

In [ ]:
FLOTORCH_API_KEY = "<YOUR FLOTORCH_API_KEY>"
FLOTORCH_BASE_URL = "<YOUR FLOTORCH_BASE_URL>" #https://gateway.flotorch.cloud
FLOTORCH_MODEL = "<YOUR FLOTORCH_MODEL_ID>"
PROVIDER_NAME = "memo-provider" #eg : memo-provider
USER_ID = "YOUR USER_ID" #eg : flotorch_user1000
APP_ID = "YOUR APP_ID"   #eg : flotorch_app1000

In [ ]:
from strands.agent.agent import Agent
from strands.tools import tool
from flotorch.strands.llm import FlotorchStrandsModel
from flotorch.strands.memory import FlotorchMemoryTool

print("✔ Imported necessary libraries successfully")

## 2. Defining a Custom Tool

To extend the agent's capabilities, we define custom tool using the `@tool` decorator from Strands. The `multiply` tool takes a comma-separated string of two integers (e.g., 'x,y') and returns their product. The tool's docstring provides a clear description, enabling the agent to understand its purpose and usage.

In [ ]:
@tool
def multiply(numbers: str) -> int:
    """
    Multiply two integers provided as a comma-separated string.

    Args:
        numbers (str): Two integers in the format 'x,y' (e.g., '3,4')

    Returns:
        int: The product of the two integers
    """
    print("Multiplying... custom tool")
    a, b = map(int, numbers.split(","))
    return a * b

print("✔ Custom multiplication tool defined successfully.")

## 3. Defining a Custom Tool

To extend the agent's capabilities, we define custom tool using the `@tool` decorator from Strands. The `addition` tool takes a comma-separated string of two integers (e.g., 'x,y') and returns their sum. The tool's docstring provides a clear description, enabling the agent to understand its purpose and usage.


In [ ]:
@tool
def addition(numbers: str) -> int:
    """
    Addition of two integers provided as a comma-separated string.

    Args:
        numbers (str): Two integers in the format 'x,y' (e.g., '3,4')

    Returns:
        int: The Addition of the two integers
    """
    print("Adding... custom tool")
    a, b = map(int, numbers.split(","))
    return a + b


print("✔ Custom Addition tool defined successfully.")

## 4. Model Configuration

We initialize the `FlotorchStrandsModel` from the custom `llm` module, which serves as the reasoning engine for the agent, enabling it to process inputs, invoke tools, and leverage memory.

In [ ]:
model = FlotorchStrandsModel(
    model_id=FLOTORCH_MODEL,
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_BASE_URL,
)

print(f"✔ Flotorch LLM model configured")

## 5. Memory Configuration

This section configures the `FlotorchMemoryTool` to enable long-term memory persistence across sessions, complementing Strands' session-based context. The tool integrates with the agent's toolset to store and retrieve persistent knowledge, enhancing context-aware interactions.

- **Memory Tool Initialization:**
  - Utilizes `FlotorchMemoryTool` to store and retrieve information across sessions, configured with   `provider_name`, `user_id`, and `app_id` for unique data organization in the Flotorch backend.

- **Memory Functionality:**:
  - Enables recall of persistent knowledge (e.g., user preferences like 'I love pizza') and dynamic updates, complementing session-based persistence with cross-session context.

- **Best Practices:**:
  - Use unique `user_id` and `app_id` to avoid data conflicts.

In [ ]:
memory = FlotorchMemoryTool(
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_BASE_URL,
    provider_name=PROVIDER_NAME,
    user_id=USER_ID,
    app_id=APP_ID,
)

tools = [multiply,addition, memory]

print(f"✔ memory initialized.")

## 6. Agent Configuration

We create a Strands agent using `Agent`, integrating the Flotorch LLM, the `add` and `multiply` tools, and the `FlotorchStrandsMemory` for long-term memory.

In [ ]:
agent = Agent(
    model=model,
    tools=tools
)

print("✔ Strands agent created successfully with session and long-term memory.")

## 8. Interactive Chat
Interactive loop for engaging with the Strands agent, using session-based and long-term memory to maintain context and recall knowledge. The agent handles arithmetic queries (e.g., "Multiply 5,6", "Addition 3,4") with tools and retrieves data via `FlotorchMemoryTool`.

- **Interaction Flow:**
  - Processes queries with Flotorch LLM, tools, and memory; persists session data; recalls prior session info.

  - Typing "exit" terminates the loop, ending the interactive session.

- **Session Persistence:**
  - Stores queries/responses as `SessionMessage` with metadata; syncs agent state; supports redaction.

- **Long-Term Memory:**
  - Uses `FlotorchMemoryTool` to access/update persistent knowledge (e.g., 'I love pizza') across sessions.



In [ ]:
while True:
    user_query = input("You: ")
    if user_query.lower().strip() == "exit":
        break

    try:
        response = agent(user_query)
        print(f"Assistant: {response}\n")
    except Exception as e:
        print(f"Error: {e}\n")

print("✔ Interactive session ended.")

## Summary

This notebook detailed the implementation of a sophisticated Strands agent equipped with both session-based and long-term memory using Flotorch's infrastructure. The agent recalls information from the current conversation and persists knowledge across sessions, enhanced by custom arithmetic tools.

### Key Achievements

- **Memory Architecture**  
  Successfully integrated two memory types:  
  - `FlotorchStrandsMemory` for persistent, long-term knowledge.

- **Enhanced Agent Intelligence**  
  The agent demonstrated its ability to perform arithmetic operations (via `multiply` and `addition` tools) and recall facts from a long-term knowledge base (e.g., 'I love pizza'), leading to informed and context-rich responses.
